# Flux 01 — scFEA demo / proof-of-concept

Cilj: empiricno preveriti izvedljivost flux pipeline (TRIM-Flux Var 2) na MAJHNEM vzorcu, PREDEN gradimo poln pipeline.

Orodje: **scFEA** (single-cell Flux Estimation Analysis, GNN, GPU). Izbrano po raziskavi izvedljivosti:
- Compass ~30 min/vzorec -> per-cell na 147k neizvedljivo
- scFEA per-cell, GPU-pospesen, dropout-robusten (korelacija >0.85 pri simuliranem dropoutu)
- ~168 metabolnih modulov za cloveka (module_gene_m168)

Kaj ta demo izmeri:
1. namestitev scFEA + odvisnosti
2. format/orientacija vhoda (scFEA hoce GENE x CELICE; nasa data_rna je CELICE x GENI -> transponiraj)
3. ali so geni IMENA (scFEA rabi gene simbole) ali Ensembl ID
4. **cas izracuna** na vzorcu (npr. 2000 celic) -> ekstrapolacija na 147k
5. oblika izhoda (flux: CELICE x ~168 modulov)


## 0. Mount + namestitev scFEA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Namesti scFEA iz GitHub + odvisnosti
%cd /content
!git clone https://github.com/changwn/scFEA.git
%cd /content/scFEA
!pip install -q torch scipy numpy pandas matplotlib magic-impute

# Preglej data/ folder — imena module/stoichiometry datotek se med verzijami RAZLIKUJEJO!
# Privzeto za cloveka: module_gene_m168.csv + cmMat_171.csv (ali podobno).
print('=== scFEA data/ datoteke (poisci module_gene in cmMat za cloveka) ===')
!ls -la data/ | grep -iE "module|cmMat|m168|171"

## 1. Poti in nalaganje RNA

scFEA potrebuje CSV: vrstice=geni (simboli), stolpci=celice.

In [ ]:
import pickle, numpy as np, pandas as pd, os, time
data_parent_folder = '/content/drive/MyDrive/Diploma/data/processed'

with open(os.path.join(data_parent_folder, 'data_rna.pkl'), 'rb') as f:
    data_rna = pickle.load(f)           # CELICE x GENI (sparse ali dense)
with open(os.path.join(data_parent_folder, 'data_labels.pkl'), 'rb') as f:
    data_labels = pickle.load(f)
print('data_rna:', data_rna.shape, type(data_rna))
print('data_labels:', data_labels.shape)

# Imena genov: scFEA rabi gene SIMBOLE (npr. CD8A), NE Ensembl ID (ENSG...).
# data_rna nima imen stolpcev ce je numpy -> poskusi naloziti seznam genov.
gene_names = None
for cand in ['combined_data_columns.npz', 'gene_names.pkl', 'var_names.pkl']:
    p = os.path.join(data_parent_folder, cand)
    if os.path.exists(p):
        if cand.endswith('.npz'):
            gene_names = np.load(p, allow_pickle=True)['cols']
        else:
            gene_names = pickle.load(open(p,'rb'))
        print('Gene imena nalozena iz', cand, '->', len(gene_names))
        break
if gene_names is None:
    print('OPOZORILO: ni shranjenih gene imen. Preveri notebook 01 ali h5 var_names.')
    print('  -> scFEA potrebuje gene simbole. Ce data_rna nima imen, jih moras pridobiti iz .h5 (adata.var_names).')

## 2. Diagnostika imen genov

So to gene simboli (CD8A) ali Ensembl ID (ENSG00000153563)?

In [ ]:
if gene_names is not None:
    sample = [str(g) for g in gene_names[:10]]
    print('Prvih 10 genov:', sample)
    is_ensembl = all(str(g).upper().startswith('ENSG') for g in gene_names[:50])
    print('Videti kot Ensembl ID?', is_ensembl)
    if is_ensembl:
        print('  -> POTREBNO: pretvori Ensembl ID v gene simbole (pybiomart/mygene) preden scFEA.')
    else:
        print('  -> Videti kot gene simboli. scFEA ok.')

## 3. Vzorec za demo (proof-of-concept)

Vzemi MAJHEN vzorec (npr. 2000 celic enega pacienta), da izmerimo cas. Ne celoten dataset.

In [ ]:
from scipy.sparse import issparse

N_DEMO = 2000                  # demo velikost
col_patient = data_labels.columns.get_loc('Patient')
# vzemi celice enega pacienta (P24) za reprezentativen vzorec
mask = (data_labels['Patient'].values == 24)
idx = np.where(mask)[0]
if len(idx) > N_DEMO:
    rng = np.random.RandomState(0)
    idx = rng.choice(idx, N_DEMO, replace=False)
print(f'Demo vzorec: {len(idx)} celic (pacient 24)')

X = data_rna[idx]
X = X.toarray() if issparse(X) else np.asarray(X)
print('X (celice x geni):', X.shape)

# scFEA hoce GENE x CELICE -> transponiraj. Z imeni vrstic (geni) in stolpcev (celice).
if gene_names is not None:
    df_in = pd.DataFrame(X.T, index=[str(g) for g in gene_names],
                         columns=[f'cell_{i}' for i in range(X.shape[0])])
else:
    df_in = pd.DataFrame(X.T, index=[f'gene_{i}' for i in range(X.shape[1])],
                         columns=[f'cell_{i}' for i in range(X.shape[0])])
print('scFEA vhod (geni x celice):', df_in.shape)

os.makedirs('/content/scfea_input', exist_ok=True)
in_csv = '/content/scfea_input/demo_expr.csv'
df_in.to_csv(in_csv)
print('Shranjeno:', in_csv)

## 4. Poženi scFEA in izmeri čas

In [ ]:
%cd /content/scFEA
os.makedirs('/content/scfea_output', exist_ok=True)

# Auto-detekcija pravih imen datotek (verzije scFEA se razlikujejo).
# moduleGene za cloveka: module_gene_m168.csv. cmMat (stoichiometry): cmMat_171.csv ali podobno.
import glob as _glob
mods = [os.path.basename(p) for p in _glob.glob('data/module_gene*m168*.csv')
        if 'mouse' not in p.lower()]
cmats = [os.path.basename(p) for p in _glob.glob('data/cmMat*.csv')
         if 'mouse' not in p.lower()]
module_file = mods[0] if mods else 'module_gene_m168.csv'
cmmat_file  = cmats[0] if cmats else 'cmMat_171.csv'
print('Uporabljam moduleGene:', module_file)
print('Uporabljam stoichiometry:', cmmat_file)

t0 = time.time()
!python src/scFEA.py \
    --data_dir data \
    --input_dir /content/scfea_input \
    --test_file demo_expr.csv \
    --moduleGene_file {module_file} \
    --stoichiometry_matrix {cmmat_file} \
    --res_dir /content/scfea_output \
    --sc_imputation True
elapsed = time.time() - t0
print(f'\n=== scFEA cas za {len(idx)} celic: {elapsed:.1f} s ===')
print(f'Ekstrapolacija na 146776 celic: ~{elapsed/len(idx)*146776/60:.1f} min (linearno, groba ocena)')

## 5. Preglej izhod (flux matrika)

In [ ]:
import glob
flux_files = glob.glob('/content/scfea_output/*flux*.csv')
print('Izhodne datoteke:', glob.glob('/content/scfea_output/*.csv'))
if flux_files:
    flux = pd.read_csv(flux_files[0], index_col=0)
    print('\nFlux matrika:', flux.shape, '(pricakovano: celice x ~168 modulov)')
    print('Moduli (prvih 10):', list(flux.columns[:10]))
    print('\nStatistika flux vrednosti:')
    print('  min/max/mean:', flux.values.min(), flux.values.max(), flux.values.mean())
    print('  delez nicelnih:', (flux.values == 0).mean())
    print('\nPrvih 5 celic x 5 modulov:')
    print(flux.iloc[:5, :5])
else:
    print('Ni najdene flux datoteke - preveri scFEA izpis zgoraj za napake.')

## 6. Zaključek demo

Ce je demo uspel, imamo potrjeno:
- scFEA tece na nasih podatkih (format/imena genov ok)
- cas na vzorec -> ekstrapolacija na cel dataset (ali je <1-2h sprejemljivo?)
- flux izhod (celice x ~168 modulov) -> to bo 3. modaliteta v TRIM

Naslednji koraki (PO uspesnem demo):
1. Pridobi/pretvori gene simbole ce so Ensembl
2. Pozeni scFEA na CELOTNEM datasetu -> flux matrika (146776 x ~168)
3. Shrani data_flux.pkl (poravnan z data_rna po vrsticah)
4. Integracija: flux encoder/decoder kot 3. modaliteta v TRIM (notebook 03 flux varianta)

OPOZORILA za diplomo:
- scFEA fluksi so RELATIVNI, model-odvisni (ne absolutne hitrosti)
- benchmark scFEA vs Compass vs METAFlux ne obstaja -> trditi 'scFEA-ocenjeni fluksi'
